# Tennis Ball Tracking with TrackNet - GPU Version
This notebook uses Google Colab GPU to run ball tracking on tennis videos.

## Setup Instructions:
1. **Runtime → Change Runtime Type → GPU**
2. In Google Drive, create a folder `tennis_coach` containing:
   - your tennis video (any `.mp4` — the notebook auto-detects it)
   - a `models/` subfolder with `tracknet_model.pt`
3. Run cells in order — the video filename and model are detected automatically.

## Notes
- The video is copied to Colab's **local disk** before reading (Drive is very slow to read frame-by-frame).
- Frames are auto-resized to **1280×720** (TrackNet's expected input) so ball coordinates are correct for any source resolution.
- The output video is saved to your Drive **and** offered as a download.

In [ ]:
# Install dependencies
!pip install opencv-python scipy torch torchvision tqdm -q
print("Dependencies installed!")

In [ ]:
import cv2
import torch
import numpy as np
from scipy.spatial import distance
from tqdm import tqdm
import os
from google.colab import drive

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Update this path to where you stored files on Google Drive
WORK_DIR = '/content/drive/My Drive/tennis_coach'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Files in directory: {os.listdir()}')

In [ ]:
# Define TrackNet Model Architecture
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, pad=1, stride=1, bias=True):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=pad, bias=bias),
            nn.ReLU(),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        return self.block(x)

class BallTrackerNet(nn.Module):
    def __init__(self, input_channels=3, out_channels=14):
        super().__init__()
        self.out_channels = out_channels
        self.input_channels = input_channels

        self.conv1 = ConvBlock(in_channels=self.input_channels, out_channels=64)
        self.conv2 = ConvBlock(in_channels=64, out_channels=64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = ConvBlock(in_channels=64, out_channels=128)
        self.conv4 = ConvBlock(in_channels=128, out_channels=128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv5 = ConvBlock(in_channels=128, out_channels=256)
        self.conv6 = ConvBlock(in_channels=256, out_channels=256)
        self.conv7 = ConvBlock(in_channels=256, out_channels=256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv8 = ConvBlock(in_channels=256, out_channels=512)
        self.conv9 = ConvBlock(in_channels=512, out_channels=512)
        self.conv10 = ConvBlock(in_channels=512, out_channels=512)
        self.ups1 = nn.Upsample(scale_factor=2)
        self.conv11 = ConvBlock(in_channels=512, out_channels=256)
        self.conv12 = ConvBlock(in_channels=256, out_channels=256)
        self.conv13 = ConvBlock(in_channels=256, out_channels=256)
        self.ups2 = nn.Upsample(scale_factor=2)
        self.conv14 = ConvBlock(in_channels=256, out_channels=128)
        self.conv15 = ConvBlock(in_channels=128, out_channels=128)
        self.ups3 = nn.Upsample(scale_factor=2)
        self.conv16 = ConvBlock(in_channels=128, out_channels=64)
        self.conv17 = ConvBlock(in_channels=64, out_channels=64)
        self.conv18 = ConvBlock(in_channels=64, out_channels=self.out_channels)

        self._init_weights()

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.pool1(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.pool2(x)
        x = self.conv5(x)
        x = self.conv6(x)
        x = self.conv7(x)
        x = self.pool3(x)
        x = self.conv8(x)
        x = self.conv9(x)
        x = self.conv10(x)
        x = self.ups1(x)
        x = self.conv11(x)
        x = self.conv12(x)
        x = self.conv13(x)
        x = self.ups2(x)
        x = self.conv14(x)
        x = self.conv15(x)
        x = self.ups3(x)
        x = self.conv16(x)
        x = self.conv17(x)
        x = self.conv18(x)
        return x

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.uniform_(module.weight, -0.05, 0.05)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

print("TrackNet model defined")

In [ ]:
# Define Ball Detector
class BallDetector:
    def __init__(self, path_model, device='cuda'):
        self.model = BallTrackerNet(input_channels=9, out_channels=256)
        self.device = device
        self.model.load_state_dict(torch.load(path_model, map_location=device))
        self.model = self.model.to(device)
        self.model.eval()
        self.width = 640
        self.height = 360
        print(f'Model loaded on {device}')

    def infer_model(self, frames):
        ball_track = [(None, None)] * 2
        prev_pred = [None, None]
        for num in tqdm(range(2, len(frames)), desc='Ball Detection'):
            img = cv2.resize(frames[num], (self.width, self.height))
            img_prev = cv2.resize(frames[num-1], (self.width, self.height))
            img_preprev = cv2.resize(frames[num-2], (self.width, self.height))
            imgs = np.concatenate((img, img_prev, img_preprev), axis=2)
            imgs = imgs.astype(np.float32) / 255.0
            imgs = np.rollaxis(imgs, 2, 0)
            inp = np.expand_dims(imgs, axis=0)

            with torch.no_grad():
                out = self.model(torch.from_numpy(inp).float().to(self.device))
            output = out.argmax(dim=1).detach().cpu().numpy()
            x_pred, y_pred = self.postprocess(output, prev_pred)
            prev_pred = [x_pred, y_pred]
            ball_track.append((x_pred, y_pred))
        return ball_track

    def postprocess(self, feature_map, prev_pred, scale=2, max_dist=80):
        feature_map *= 255
        feature_map = feature_map.reshape((self.height, self.width))
        feature_map = feature_map.astype(np.uint8)
        ret, heatmap = cv2.threshold(feature_map, 127, 255, cv2.THRESH_BINARY)
        circles = cv2.HoughCircles(heatmap, cv2.HOUGH_GRADIENT, dp=1, minDist=1,
                                   param1=50, param2=2, minRadius=2, maxRadius=7)
        x, y = None, None
        if circles is not None:
            if prev_pred[0]:
                for i in range(len(circles[0])):
                    x_temp = circles[0][i][0] * scale
                    y_temp = circles[0][i][1] * scale
                    dist = distance.euclidean((x_temp, y_temp), prev_pred)
                    if dist < max_dist:
                        x, y = x_temp, y_temp
                        break
            else:
                x = circles[0][0][0] * scale
                y = circles[0][0][1] * scale
        return x, y

print("BallDetector class defined")

In [ ]:
# Helper functions
TARGET_W, TARGET_H = 1280, 720  # TrackNet expects 1280x720 for correct ball coords

def read_video(path_video, target_size=(TARGET_W, TARGET_H)):
    """Read all frames from video, resizing to target_size (TrackNet needs 1280x720)."""
    cap = cv2.VideoCapture(path_video)
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    pbar = tqdm(total=total if total > 0 else None, desc='Reading frames')
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if target_size is not None and (frame.shape[1], frame.shape[0]) != target_size:
            frame = cv2.resize(frame, target_size)
        frames.append(frame)
        pbar.update(1)
    pbar.close()
    cap.release()
    return frames, fps

def visualize_and_save(frames, ball_track, fps, output_path):
    """Create output video with ball tracking visualization"""
    height, width = frames[0].shape[:2]
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    for i, frame in enumerate(tqdm(frames, desc='Writing Video')):
        img_res = frame.copy()
        if i < len(ball_track) and ball_track[i][0] is not None:
            x, y = int(ball_track[i][0]), int(ball_track[i][1])
            img_res = cv2.circle(img_res, (x, y), radius=5, color=(0, 255, 0), thickness=2)
            img_res = cv2.putText(img_res, 'ball', org=(x + 8, y + 8),
                                  fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.8,
                                  thickness=2, color=(0, 255, 0))
        out.write(img_res)
    out.release()
    print(f"\nVideo saved to {output_path}")

print("Helper functions defined")

In [ ]:
# Load Model and Video
import glob, shutil

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# --- Model path (auto-detect any .pt in models/) ---
model_candidates = glob.glob('models/*.pt')
assert model_candidates, "No .pt model found in models/ folder on your Drive!"
model_path = model_candidates[0]
print(f'Model: {model_path}')

# --- Video path (auto-detect any .mp4 in the working dir) ---
video_candidates = sorted(glob.glob('*.mp4'))
# ignore any previously generated output videos
video_candidates = [v for v in video_candidates if not v.startswith('output_')]
assert video_candidates, "No input .mp4 found in your tennis_coach folder!"
drive_video = video_candidates[0]
print(f'Found video on Drive: {drive_video}')

output_path = 'output_ball_tracking_gpu.mp4'

# --- Copy video from Drive to fast local disk (Drive reads are very slow) ---
local_video = '/content/input_video.mp4'
print('Copying video to local disk for fast reading...')
shutil.copy(drive_video, local_video)
print(f'Copied to {local_video} ({os.path.getsize(local_video)/1e6:.1f} MB)')

# --- Quick metadata check (instant, no full load) ---
cap = cv2.VideoCapture(local_video)
src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
src_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
src_n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
src_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print(f'Source video: {src_w}x{src_h}, {src_n} frames @ {src_fps:.1f} fps')
if (src_w, src_h) != (TARGET_W, TARGET_H):
    print(f'  -> will be resized to {TARGET_W}x{TARGET_H} for TrackNet')

# --- Load all frames (resized to 1280x720) ---
print('\nLoading video frames...')
frames, fps = read_video(local_video)
print(f'Loaded {len(frames)} frames at {fps} fps')
print(f'Frame shape: {frames[0].shape}')

In [ ]:
# Run Ball Detection
print("Initializing BallDetector...")
ball_detector = BallDetector(model_path, device)

print("\nRunning ball detection...")
ball_track = ball_detector.infer_model(frames)

# Count detections
detected_count = sum(1 for x, y in ball_track if x is not None)
print(f"\nBall detected in {detected_count}/{len(ball_track)} frames ({100*detected_count/len(ball_track):.1f}%)")

In [ ]:
# Generate Output Video
local_output = '/content/' + output_path
print("Generating output video with ball tracking...")
visualize_and_save(frames, ball_track, fps, local_output)

# Copy result back to Drive so it persists
shutil.copy(local_output, output_path)
print(f"\nOutput video on Drive: {output_path}")
print(f"File size: {os.path.getsize(local_output) / 1e6:.1f} MB")

In [ ]:
# Download Results (also already saved to your Drive)
from google.colab import files
print("Downloading output video...")
files.download(local_output)

In [ ]:
# Optional: Save detection data as CSV
import csv

csv_path = 'ball_tracking_data.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['frame', 'x', 'y'])
    for i, (x, y) in enumerate(ball_track):
        writer.writerow([i, x, y])

print(f"Ball tracking data saved to {csv_path}")
files.download(csv_path)